# 01. Getting Started with VAFT

This session introduces the everyday VAFT workflow on a single VEST discharge:

```text
load an ODS -> inspect its IDS roots -> look at the geometry
            -> plot diagnostics -> look at the equilibrium
```

Everything here runs offline from data packaged with VAFT. No database access or
credentials are needed.

## What is VAFT

**VAFT** is the analysis interface for VEST data. It loads a discharge, gives
you its measurements in a standard structure, and plots them.

**VEST** is the spherical tokamak at Seoul National University. One **shot** is
one discharge: a few hundred milliseconds during which coil currents ramp, a
plasma forms, and every diagnostic records a signal.

**OMAS / ODS** is how that data is held in memory. An **ODS** behaves like a
dictionary whose keys are dotted paths:

```python
ods["magnetics.ip.0.data"]
```

**IDS** is the vocabulary those paths use. It comes from IMAS, the ITER data
standard, so an ODS is organised the same way for VEST as it would be for any
other tokamak. Each top-level name -- `magnetics`, `pf_active`, `equilibrium`,
`wall` -- is one **IDS**: a family of related measurements.

Two things to carry forward, and they are deliberately different:

- **You find *data* by IDS path.** The plasma current is stored at
  `magnetics.ip.0.data`, because that is where IMAS puts it.
- **You find *plots* by what the thing physically is.** The function that draws
  the plasma current is `vaft.omas.plot_plasma_current_time`, not
  `plot_magnetics_...`.

Storage location and physical identity are not the same question, and VAFT
stopped pretending they were. The next sections use both.

## Imports

Two imports are enough for this session.

- **`vaft`** is the analysis interface. Everything in this notebook is reached
  through it: `vaft.omas` for loading and plotting an ODS, `vaft.database` for
  fetching a shot from the VEST database.
- **`matplotlib.pyplot`** is the plotting library VAFT draws with. We use it
  here for exactly one thing -- `plt.show()`, which displays the figure a VAFT
  plotting function just produced.

You do not need NumPy, Pandas, or any VAFT submodule imported by hand.

In [ ]:
import vaft
import matplotlib.pyplot as plt


## Loading Data

There are two ways to get an ODS, and you will use both over the course.

### Packaged sample

VAFT ships a complete VEST discharge, shot 39915, inside the package itself.
`sample_ods()` returns it with no arguments, no configuration, and no network
access, so every example in this notebook is reproducible on any machine.

This is the right starting point for learning, for testing an idea, and for
anything you want a colleague to be able to re-run.

In [ ]:
ods = vaft.omas.sample_ods()


### Loading a VEST shot

Real work uses the VEST database. The call is one line and returns the same kind
of ODS as the packaged sample, so everything you learn below applies unchanged:

```python
ods = vaft.database.load(41672)
```

This one does need access: the database is reached over HSDS, which reads your
credentials from `~/.hscfg`. Run `hsconfigure` once to write that file (see
`install/README.md` for the endpoint and how to request an account). Until then
`vaft.database.load` will not connect, which is why the rest of this notebook
stays on the packaged sample.

## ODS Structure

Before plotting anything, look at what the discharge actually contains. The
top-level keys are the IDS roots -- the families of data present for this shot.

In [ ]:
sorted(ods.keys())


Reading that list:

| IDS | What it holds |
| --- | --- |
| `magnetics` | plasma current, flux loops, magnetic probes, diamagnetic flux |
| `pf_active` | poloidal field coil currents -- the programmed waveforms |
| `tf` | toroidal field coil current and the resulting field |
| `spectrometer_uv` | filterscope line emission, including H-alpha |
| `barometry` | neutral pressure gauges |
| `wall` | vessel and limiter outline |
| `pf_passive` | passive conducting structure |
| `equilibrium` | the reconstructed magnetic configuration |
| `em_coupling` | mutual inductances between coils, structure and diagnostics |
| `dataset_description` | shot number and provenance |

Not every shot has every IDS. A diagnostic that did not run, or has not been
processed, is simply absent -- which is why checking the keys first is a habit
worth having.

## Geometry Overview

A signal means little until you know where it was measured. VAFT composes the
machine and its diagnostics into two orientation views.

### Poloidal view (R-Z)

This is the cross-section you will see most often: the limiter outline, the PF
coils, the passive structure, and the magnetic diagnostic positions -- flux
loops and B-field probes shown separately -- all in one axes.

In [ ]:
vaft.omas.plot_machine_geometry_poloidal(ods)
plt.show()


### Top view

The same machine seen from above: the limiter's inboard and outboard extent,
with the plasma's extent from the equilibrium reconstruction inside it.

Look at the inboard side. The plasma reaches the limiter rather than stopping
short of it -- this discharge is *limited* on the centre stack, which is how
VEST normally operates. VEST's wall description carries a limiter outline and no
separate vacuum vessel, which is why these rings are labelled limiter.

Between the two views you should now be able to place any signal in this
notebook: the poloidal view tells you where a diagnostic sits in the
cross-section, the top view tells you how far the plasma reaches in major
radius.

In [ ]:
vaft.omas.plot_machine_geometry_topview(ods)
plt.show()


## How VAFT Names Its Plots

Every plotting function is `vaft.omas.plot_` followed by a canonical name built
from up to three parts:

```text
{subject}_{view}[_{quantity}]
```

- **subject** -- what the data physically *is*: `plasma_current`, `flux_loop`,
  `pf_coil`, `equilibrium`, `machine`.
- **view** -- how it is drawn: `time`, `profile`, `evolution`, `field`,
  `geometry`, `geometry3d`, `spectrum`, `spectrogram`, `overview`, `image`,
  `animation`.
- **quantity** -- which one, when a subject has several: `flux_loop_time_flux`
  and `flux_loop_time_voltage` are the same probes measuring different things.

So `plasma_current_time` is the plasma current, drawn against time. You can read
the name before you read the documentation, which is the point.

**The subject is not the IDS.** `plasma_current_time` reads the `magnetics` IDS;
`pf_coil_time_current` reads `pf_active`. Storage location is recorded
separately, as the plot's `domain`. A beginner does not need to track that --
you name what you want to see, not where it is kept.

If you know a plot by an older, IDS-shaped name such as `magnetics_time_ip`, it
still works and warns, telling you the new name. Those aliases are removed in
version 0.8.0.

### Finding a plot

`vaft.omas.available_plots(ods)` answers "what can *this* shot show me?" -- it
filters the full catalogue down to the plots whose data the object actually
holds.

In [ ]:
rows = vaft.omas.available_plots(ods)
print(f"{len(rows)} plots are available for this shot")

for row in rows:
    if row["subject"] in ("plasma_current", "pf_coil", "barometry"):
        print(f"  {row['name']:<32} subject={row['subject']:<16} stored in {row['domain']}")

## Diagnostics

Every plot below follows the same shape:

```python
vaft.omas.plot_<subject>_<view>(ods)
plt.show()
```

That is the whole interface: a canonical name, the ODS you loaded, and nothing
else. The functions return `(figure, axes)` and draw nothing until you ask, so
`plt.show()` is what puts the figure on screen.

### Plasma Current

The plasma current, measured by a Rogowski coil, is the single most important
trace on a tokamak. It tells you whether there was a plasma at all, when it
formed, how large it got, and when it ended.

Look for the interval where the current is sustained: that is the plasma phase,
and it sets the time window every other diagnostic should be read against.

In [ ]:
vaft.omas.plot_plasma_current_time(ods)
plt.show()


### PF Coil Currents

The poloidal field coils are the actuators. Their currents are programmed before
the shot and they create the field that breaks down the gas, drives the plasma
current, and holds the plasma in position.

These are causes, not effects. Compare their timing with the plasma current
above: the coils move first.

In [ ]:
vaft.omas.plot_pf_coil_time_current(ods)
plt.show()


### Toroidal Field

The toroidal field coil produces the field along the torus. On a spherical
tokamak like VEST it is comparatively weak, which is part of what makes the
configuration interesting.

It is roughly constant across the plasma phase, so it mostly serves as a
reference condition for the shot rather than as a dynamic signal. Note the axis
reads tesla here while the magnetic probes below read millitesla -- the next
section explains why.

In [ ]:
vaft.omas.plot_tf_coil_time_b_t(ods)
plt.show()


### Flux Loops

Flux loops are single turns of wire around the vessel. Each measures the poloidal
magnetic flux threading it, and together they constrain where the plasma sits.

They are one of the two magnetic measurements the equilibrium reconstruction is
fitted to. This shot has eleven; we plot the first three, since eleven nearly
identical traces teach nothing extra.

In [ ]:
vaft.omas.plot_flux_loop_time_flux(ods, channels=[0, 1, 2])
plt.show()


### Poloidal Field Probes

The other magnetic measurement: small coils that sense the local poloidal field
at fixed points around the cross-section. This shot has 76 of them, and their
positions are the "B-field Probes" markers in the poloidal geometry view.

Again three representative channels.

In [ ]:
vaft.omas.plot_b_field_probe_time_field(ods, channels=[0, 1, 2])
plt.show()


### Diamagnetic Flux

A plasma pushes back against the field that confines it, slightly reducing the
toroidal flux through the vessel. That small difference is the diamagnetic flux,
and it is a direct measure of the stored thermal energy.

It is a hard measurement -- a small change on top of a large field -- so treat
it as indicative rather than precise.

In [ ]:
vaft.omas.plot_diamagnetic_flux_time(ods)
plt.show()


### H-alpha Spectroscopy

H-alpha is light emitted when neutral hydrogen atoms are excited by the plasma.
It is measured by a filterscope looking at the plasma edge.

Because neutrals live at the edge, H-alpha reports on recycling and fuelling
there. It rises sharply at breakdown and tracks how strongly the plasma is
interacting with the wall.

In [ ]:
vaft.omas.plot_spectrometer_uv_time_intensity(ods)
plt.show()


### Barometry

The barometry gauge measures neutral gas pressure in the vessel. The pressure
before the shot -- the prefill -- is one of the few knobs an operator sets by
hand, and it strongly influences whether breakdown succeeds.

Read this one as the initial condition of the discharge.

In [ ]:
vaft.omas.plot_barometry_time_pressure(ods)
plt.show()


## Reading the Axes

By now you have seen four different units without being told about any of them:
the plasma current in **kA**, the flux loops in **mWb**, the magnetic probes in
**mT**, and the neutral pressure in **Torr**. None of those is how the data is
stored -- IMAS keeps amperes, webers, tesla and pascals.

VAFT applies one display policy to every plot:

- **The stored value never changes.** `ods["magnetics.ip.0.data"]` is in amperes
  whatever the axis says.
- **A readable display unit is chosen per quantity**, so a VEST discharge reads
  as `84.1 kA` rather than `8.41e+04 A`. Some subjects override the default
  where physics makes another unit natural: probes are shown in mT, but the
  toroidal field coil is shown in T, because those are the scales they live at.
- **Unit and scaling always move together.** This is the part that matters. A
  label saying `kA` is a promise that the numbers beside it were divided by
  1000. They cannot drift apart.
- **An unsupported unit is refused, not ignored.** Asking for a unit that makes
  no sense for the quantity raises an error rather than silently relabelling
  correct numbers with a wrong name.

You override the unit with `yunit=`. The cell below shows the same trace three
ways: the default, forced to amperes, and a request that is refused.

The figure title carries the shot number the same way -- `#39915` comes from the
dataset's own provenance, so a plot is self-identifying once it leaves your
screen.

In [ ]:
figure, axes = vaft.omas.plot_plasma_current_time(ods)
print("default:", axes.get_ylabel())

figure, axes = vaft.omas.plot_plasma_current_time(ods, yunit="A")
print("yunit='A':", axes.get_ylabel())

try:
    vaft.omas.plot_plasma_current_time(ods, yunit="furlongs")
except ValueError as error:
    print("refused:", error)

plt.close("all")

## Equilibrium

The diagnostics above are measurements. The **equilibrium** is what is inferred
from them: a reconstruction, fitted to the magnetic measurements, of where the
magnetic surfaces are and what the plasma is doing inside them.

It answers two questions:

- **the magnetic configuration**, through the poloidal flux psi; and
- **the plasma profiles**, through quantities such as pressure, current density
  and safety factor.

### Poloidal Flux (psi)

Contours of constant psi are the nested magnetic flux surfaces the plasma is
confined on. The innermost contour is the magnetic axis; the outermost closed
one is the boundary of the confined plasma.

In [ ]:
vaft.omas.plot_equilibrium_field_psi(ods)
plt.show()


### Equilibrium Profiles

Where psi shows the shape, the 1-D profiles show the state of the plasma along
it. Three matter most, plotted here together:

- **pressure `p`** -- how much thermal energy is confined, falling from the hot
  core to the cold edge;
- **current density `j`** -- how the plasma current is distributed across the
  radius; and
- **safety factor `q`** -- how many toroidal turns a field line makes per
  poloidal turn. Low `q` regions are where MHD instabilities tend to live.

Each is plotted against the normalised radius, running 0 at the magnetic axis to
1 at the boundary.

This reconstruction stores `p` and `q` but not `j`, so two panels are drawn. That
is the normal behaviour of a VAFT overview: it plots the quantities the data
actually contains rather than leaving an empty panel.

In [ ]:
vaft.omas.plot_equilibrium_overview_profiles(ods)
plt.show()


## Exercise

Two things to try. Neither is checked by the notebook -- edit the cell below and
run it.

1. **Plot a diagnostic on its own.** Pick one profile from the overview above
   and plot it by itself, for example `vaft.omas.plot_equilibrium_profile_q`.
   Every function used in this notebook has a single-quantity sibling.

2. **Load a different shot.** Once HSDS access is configured, load another shot
   with `vaft.database.load` and plot its plasma current. Compare it with shot
   39915: is the current larger, does it last longer, when does it form?

In [ ]:
# 1. Plot one profile on its own.
# vaft.omas.plot_equilibrium_profile_q(ods)
# plt.show()

# 2. Load another shot and plot its plasma current. Needs HSDS access.
# other = vaft.database.load(41672)
# vaft.omas.plot_plasma_current_time(other)
# plt.show()


## Summary

The workflow, start to finish:

```python
import vaft
import matplotlib.pyplot as plt

ods = vaft.omas.sample_ods()          # or vaft.database.load(shot)
sorted(ods.keys())                    # which IDS does this shot have?
vaft.omas.available_plots(ods)        # which plots can it produce?

vaft.omas.plot_machine_geometry_poloidal(ods)   # where things are
vaft.omas.plot_plasma_current_time(ods)         # what was measured
vaft.omas.plot_equilibrium_field_psi(ods)       # what was reconstructed
plt.show()
```

That is: **load an ODS, inspect its IDS roots, look at the geometry, plot the
diagnostics, look at the equilibrium.**

Three habits are worth keeping:

- **Name plots by subject, not by storage.** `{subject}_{view}[_{quantity}]`
  reads as physics, and `available_plots(ods)` turns "what is there?" into a
  list rather than a guess.
- **Read the unit on the axis.** It is chosen for legibility and it always
  matches the numbers beside it; `yunit=` overrides it when you need a
  comparison on fixed scales.
- **Check the geometry before trusting a trace.** A signal without a position is
  hard to interpret and easy to over-read.

Session 02 continues from here, connecting these traces to how the discharge was
operated and to the vacuum fields the coils produce.

## Additional Resources

Everything above is VAFT. The links below are for going deeper into the
standards VAFT is built on -- IMAS, OMAS, and the Data Dictionary that defines
the paths you have been reading. None of it is needed to finish this session.

### Tutorials and hands-on examples

Concrete examples and exercises for IMAS/OMAS themselves, beyond the VAFT
workflow:

- [IMAS Tutorial](https://github.com/iterorganization/IMAS-tutorial)
- [IMAS-Python 101](https://imas-python.readthedocs.io/en/stable/courses/basic_user_training.html)
- [Advanced IMAS-Python](https://imas-python.readthedocs.io/en/stable/courses/advanced_user_training.html)
- [OMAS Examples Gallery](https://gafusion.github.io/omas/auto_examples/index.html)

### API documentation by language

The same data structures are reachable from several languages. VAFT is Python
and uses OMAS, so the first group is the one that applies here.

**Python**

- [OMAS](https://gafusion.github.io/omas/)
- [IMAS-Python](https://imas-python.readthedocs.io/en/stable/)
- [IMAS-Python API Reference](https://imas-python.readthedocs.io/en/stable/api.html)

**MATLAB**

- [IMAS-MATLAB](https://imas-matlab.readthedocs.io/)

**Julia**

- [IMASdd.jl](https://projecttorreypines.github.io/IMASdd.jl/)

### IMAS Data Dictionary and schema references

This tutorial showed you a handful of paths such as `magnetics.ip.0.data`. When
you need the rest -- the full IDS hierarchy, node names, coordinates, units,
metadata and conventions -- read the Data Dictionary rather than guessing from
examples.

- [OMAS schema browser -- IMAS Data Dictionary 3.41.0](https://gafusion.github.io/omas/schema.html)
- [Official IMAS Data Dictionary 3.41.0](https://imas-data-dictionary.readthedocs.io/en/3.41.0-doc/)
- [Official IMAS Data Dictionary 4.1.1 (latest)](https://imas-data-dictionary.readthedocs.io/en/4.1.1/)

**Which version applies here.** VAFT works in the Data Dictionary version
supported through OMAS, currently **3.41.0**. Paths and examples in this
tutorial follow that representation unless stated otherwise. The 4.1.1 link is
included as a pointer to the current Data Dictionary, not as the version this
tutorial describes -- node names and coordinates do move between major versions,
so check which one a document refers to before copying a path from it.

The ODS you loaded reports its own version, so you never have to take that on
trust:

In [ ]:
ods.imas_version